# UCS420 Cognitive Computing — Assignment 4
## A Cognitive FAQ System Using Pandas (Nova 2.0)

This notebook implements Q1–Q6 from the assignment.

**Enter your own college roll number in Q1.** The last two digits are automatically used to create the two personalized FAQ entries.

## Import Pandas

In [ ]:
import pandas as pd

## Q1. Build Your Personalized Knowledge Base

The assignment requires exactly 6 FAQ entries:
- 4 fixed entries
- 2 personalized entries based on the last two digits of your roll number

For each last digit `d`:
`category = ["billing", "account", "general"][d % 3]`

In [ ]:
# Enter your own college roll number here
roll_number = input("Enter your college roll number: ").strip()

if not roll_number.isdigit():
    raise ValueError("Please enter the roll number using digits only.")

last_two_digits = roll_number[-2:]
print("Last two digits:", last_two_digits)

In [ ]:
categories = ["billing", "account", "general"]

fixed_entries = [
    {"question": "what is the annual fee",
     "answer": "The annual fee is Rs 500.",
     "keywords": "fee cost price charge", "category": "billing"},
    {"question": "how to reset password",
     "answer": "Go to Settings > Reset Password.",
     "keywords": "password reset login", "category": "account"},
    {"question": "what are your working hours",
     "answer": "We are open 9 AM to 5 PM.",
     "keywords": "hours timing open time", "category": "general"},
    {"question": "how can i pay the fee",
     "answer": "You can pay via UPI, card, or net banking.",
     "keywords": "pay payment upi fee", "category": "billing"}
]

templates = {
    "billing": {
        "question": "how can i check my payment receipt",
        "answer": "You can view or download your payment receipt from the billing section.",
        "keywords": "receipt invoice payment"},
    "account": {
        "question": "how do i update my registered mobile number",
        "answer": "Go to Account Settings and update your registered mobile number.",
        "keywords": "mobile number update profile"},
    "general": {
        "question": "where can i find help for a common issue",
        "answer": "You can visit the Help section to find guidance for common issues.",
        "keywords": "help support guidance"}
}

personalized_entries = []
for digit in last_two_digits:
    d = int(digit)
    category = categories[d % 3]
    t = templates[category]
    personalized_entries.append({
        "question": t["question"],
        "answer": t["answer"],
        "keywords": t["keywords"],
        "category": category
    })

faq_df = pd.DataFrame(fixed_entries + personalized_entries)

print("Final 6-row DataFrame:")
display(faq_df)

## Q2. Generate and Score a Hypothesis

In [ ]:
def score_query(query, df):
    query_words = set(query.lower().split())
    results = []

    for index, row in df.iterrows():
        keywords = set(row["keywords"].lower().split())
        matched = query_words.intersection(keywords)
        score = len(matched)
        if score > 0:
            results.append({
                "index": index,
                "question": row["question"],
                "answer": row["answer"],
                "category": row["category"],
                "matched_keywords": ", ".join(sorted(matched)),
                "confidence": score
            })

    result_df = pd.DataFrame(results)
    if result_df.empty:
        return result_df
    return result_df.sort_values("confidence", ascending=False).reset_index(drop=True)

query = input("Enter an FAQ query: ")
display(score_query(query, faq_df))

## Q3. Find FAQs in the Same Category

In [ ]:
def same_category(category_name, df):
    return df[df["category"].str.lower() == category_name.lower()]

personalized_category = personalized_entries[0]["category"]
print("Category selected:", personalized_category)
display(same_category(personalized_category, faq_df))

## Q4. Add a New Keyword and Save the Updated DataFrame

Pick one FAQ entry, enter a new keyword, and save the complete DataFrame as `<your_roll_number>_faq_data.csv`.

In [ ]:
print("Available FAQ entries:")
for i, question in enumerate(faq_df["question"]):
    print(f"{i}: {question}")

entry_number = int(input("\nEnter entry number (0-5): "))
new_keyword = input("Enter a new keyword: ").strip().lower()

if not 0 <= entry_number < len(faq_df):
    raise ValueError("Entry number must be between 0 and 5.")
if not new_keyword:
    raise ValueError("Keyword cannot be empty.")

existing = faq_df.loc[entry_number, "keywords"].split()
if new_keyword not in existing:
    faq_df.loc[entry_number, "keywords"] += " " + new_keyword

output_filename = f"{roll_number}_faq_data.csv"
faq_df.to_csv(output_filename, index=False)

print("Updated DataFrame:")
display(faq_df)
print("Saved as:", output_filename)

## Q5. Count FAQ Entries Per Category Using `groupby`

In [ ]:
category_counts = faq_df.groupby("category").size()
display(category_counts)

## Q6. Modified Scoring Function — Show All Highest-Scoring Ties

If two or more entries tie for the highest score, all tied entries are displayed.

In [ ]:
def score_query_with_ties(query, df):
    query_words = set(query.lower().split())
    results = []

    for index, row in df.iterrows():
        keywords = set(row["keywords"].lower().split())
        matched = query_words.intersection(keywords)
        score = len(matched)
        if score > 0:
            results.append({
                "index": index,
                "question": row["question"],
                "answer": row["answer"],
                "category": row["category"],
                "matched_keywords": ", ".join(sorted(matched)),
                "confidence": score
            })

    result_df = pd.DataFrame(results)
    if result_df.empty:
        print("No matching FAQ entry found.")
        return result_df

    highest = result_df["confidence"].max()
    top_matches = result_df[result_df["confidence"] == highest].reset_index(drop=True)

    if len(top_matches) > 1:
        print("Tie detected — all highest-scoring entries:")
    else:
        print("No tie — highest-scoring entry:")
    display(top_matches)
    return top_matches

### Q6 Demonstration 1 — Query Producing a Tie

In [ ]:
# Both fixed billing questions contain the keyword "fee"
tie_query = "fee"
print("Query:", tie_query)
score_query_with_ties(tie_query, faq_df)

### Q6 Demonstration 2 — Query Without a Tie

In [ ]:
non_tie_query = "password reset login"
print("Query:", non_tie_query)
score_query_with_ties(non_tie_query, faq_df)